# 01 · Portfolio Overview

Hands-on walkthrough of the portfolio Python bindings, adapted from `examples/scripts/portfolio/portfolio_example.py`. Build entities and positions, value a portfolio, aggregate metrics, and try a rate shock scenario.



### Learning Objectives
- Set up reusable market data for portfolio valuation
- Build entities, instruments, and positions with the fluent `PortfolioBuilder`
- Value portfolios and aggregate metrics by position and entity
- Demonstrate multi-asset portfolios (rates + equities) and scenario shocks
- Inspect position units and long/short handling



In [1]:
from datetime import date

from finstack.core.currency import Currency
from finstack.core.dates.daycount import DayCount
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve, HazardCurve
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.money import Money
from finstack.portfolio import Entity, PortfolioBuilder, Position, PositionUnit, aggregate_metrics, value_portfolio
from finstack.valuations.instruments import Bond, CreditDefaultSwap, Deposit, Equity, InterestRateSwap


as_of = date(2024, 1, 2)


def build_market_data(as_of: date) -> MarketContext:
    market = MarketContext()

    usd_curve = DiscountCurve(
        "USD-OIS",
        as_of,
        [
            (0.0, 1.0),
            (0.5, 0.9975),
            (1.0, 0.9950),
            (3.0, 0.9750),
            (5.0, 0.9500),
            (10.0, 0.9000),
        ],
    )
    market.insert_discount(usd_curve)

    forward_curve = ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.0450), (1.0, 0.0475), (3.0, 0.0500), (5.0, 0.0525)],
        base_date=as_of,
    )
    market.insert_forward(forward_curve)

    hazard_curve = HazardCurve(
        "ACME-HZD",
        as_of,
        [(0.0, 0.0120), (3.0, 0.0180), (5.0, 0.0220)],
        recovery_rate=0.40,
        par_points=[
            (0.25, 140.0),
            (0.50, 145.0),
            (1.0, 150.0),
            (2.0, 160.0),
            (3.0, 170.0),
            (5.0, 190.0),
            (7.0, 200.0),
            (10.0, 210.0),
            (15.0, 220.0),
            (20.0, 225.0),
            (30.0, 230.0),
        ],
    )
    market.insert_hazard(hazard_curve)

    market.insert_price("AAPL-SPOT", MarketScalar.price(Money(185.0, "USD")))
    market.insert_price("MSFT-SPOT", MarketScalar.price(Money(420.0, "USD")))

    return market


market = build_market_data(as_of)
base_ccy = Currency("USD")



## 1. Build a basic portfolio
Create entities, instruments, and positions (bond, deposit, swap, CDS), then assemble a portfolio with the fluent builder.



In [2]:
entity_corp = Entity("CORP_A").with_name("Corporate A").with_tag("sector", "Finance")
entity_fund = Entity("FUND_B").with_name("Fund B").with_tag("sector", "Technology")

bond = Bond.fixed_semiannual(
    "BOND_CORP_A",
    Money(5_000_000, "USD"),
    0.045,
    date(2024, 1, 15),
    date(2029, 1, 15),
    "USD-OIS",
)

deposit = Deposit(
    "DEPOSIT_MM",
    Money(2_000_000, "USD"),
    as_of,
    date(2024, 7, 2),
    DayCount.ACT_360,
    "USD-OIS",
    quote_rate=0.0525,
)

swap = InterestRateSwap.usd_receive_fixed(
    "IRS_USD_5Y",
    Money(10_000_000, "USD"),
    0.0425,
    date(2024, 1, 5),
    date(2029, 1, 5),
)

cds = CreditDefaultSwap.buy_protection(
    "CDS_CORP_A_5Y",
    Money(5_000_000, "USD"),
    150.0,
    as_of,
    date(2029, 1, 2),
    "USD-OIS",
    "ACME-HZD",
)

positions = [
    Position("POS_001", "CORP_A", "BOND_CORP_A", bond, 1.0, PositionUnit.UNITS),
    Position("POS_002", "FUND_B", "DEPOSIT_MM", deposit, 1.0, PositionUnit.UNITS),
    Position("POS_003", "CORP_A", "IRS_USD_5Y", swap, 1.0, PositionUnit.notional()),
    Position("POS_004", "CORP_A", "CDS_CORP_A_5Y", cds, 1.0, PositionUnit.UNITS),
]

portfolio = (
    PortfolioBuilder("MULTI_ASSET_FUND")
    .name("Multi-Asset Investment Fund")
    .base_ccy(base_ccy)
    .as_of(as_of)
    .entity([entity_corp, entity_fund])
    .position(positions)
    .tag("strategy", "balanced")
    .build()
)

portfolio.validate()

print(portfolio)
print(f"Entities: {len(portfolio.entities)} | Positions: {len(portfolio.positions)}")



Multi-Asset Investment Fund
Entities: 2 | Positions: 4


## 2. Value the portfolio and inspect metrics
Use the helper functions to value the portfolio and retrieve both position-level and aggregated metrics.



In [3]:
valuation = value_portfolio(portfolio, market)

print(f"Total base value: {valuation.total_base_ccy.format()}")
print(f"Positions priced: {len(valuation.position_values)}")

print("\nPosition values:")
for pos_id, pos_value in valuation.position_values.items():
    print(f"  {pos_id}: native={pos_value.value_native.format()} | base={pos_value.value_base.format()}")

metrics = aggregate_metrics(valuation)
print(f"\nAggregated metrics: {len(metrics.aggregated)} available")
for metric_id in list(metrics.aggregated.keys()):
    metric = metrics.get_metric(metric_id)
    if metric:
        print(f"  {metric.metric_id}: {metric.total:.6f}")



Total base value: USD 5447002.90
Positions priced: 4

Position values:
  POS_001: native=USD 5845508.30 | base=USD 5845508.30
  POS_002: native=USD 47964.69 | base=USD 47964.69
  POS_003: native=USD -369576.57 | base=USD -369576.57
  POS_004: native=USD -76893.52 | base=USD -76893.52

Aggregated metrics: 54 available
  theta: -54.355571
  dv01: -7649.474437
  bucketed_dv01: -7153.194437
  cs01: 2267.840000
  bucketed_cs01: 2268.430000
  delta: 0.000000
  gamma: 0.000000
  vega: 0.000000
  rho: 0.000000
  pv01: 2360.222514
  bucketed_dv01::6m: -105.794437
  bucketed_dv01::7y: 0.000000
  bucketed_dv01::10y: -37.870000
  bucketed_dv01::15y: 0.000000
  bucketed_dv01::3y: -75.700000
  bucketed_dv01::30y: 0.000000
  bucketed_dv01::2y: 0.000000
  bucketed_dv01::1y: -17.720000
  bucketed_dv01::3m: 0.000000
  bucketed_dv01::20y: 0.000000
  bucketed_dv01::5y: -2454.690000
  bucketed_dv01::USD-OIS::3m: 0.000000
  bucketed_dv01::USD-OIS::15y: 0.000000
  bucketed_dv01::USD-SOFR-3M::7y: 0.000000
  b

## 3. Multi-entity, multi-asset portfolio
Mix rates, deposits, equities, and swaps across entities to demonstrate grouping by entity.



In [4]:
from finstack.core.config import FinstackConfig

entities = [
    Entity("CORP_TREASURY").with_name("Corporate Treasury").with_tag("department", "treasury"),
    Entity("EQUITY_DESK").with_name("Equity Trading Desk").with_tag("department", "trading"),
    Entity("DERIVATIVES").with_name("Derivatives Desk").with_tag("department", "trading"),
]

bond_multi = Bond.fixed_semiannual(
    "CORP_BOND",
    Money(4_000_000, "USD"),
    0.048,
    date(2024, 1, 15),
    date(2028, 1, 15),
    "USD-OIS",
)

deposit_multi = Deposit(
    "MM_DEPOSIT",
    Money(1_500_000, "USD"),
    as_of,
    date(2024, 3, 2),
    DayCount.ACT_360,
    "USD-OIS",
    quote_rate=0.0475,
)

equity_aapl = Equity.create("AAPL_POS", "AAPL", base_ccy, shares=5_000.0)
equity_msft = Equity.create("MSFT_POS", "MSFT", base_ccy, shares=2_000.0)

swap_multi = InterestRateSwap.usd_pay_fixed(
    "IRS_PAY_FIXED",
    Money(8_000_000, "USD"),
    0.0450,
    date(2024, 1, 5),
    date(2027, 1, 5),
)

positions_multi = [
    Position("POS_BOND", "CORP_TREASURY", "CORP_BOND", bond_multi, 1.0, PositionUnit.UNITS),
    Position("POS_DEP", "CORP_TREASURY", "MM_DEPOSIT", deposit_multi, 1.0, PositionUnit.UNITS),
    Position("POS_AAPL", "EQUITY_DESK", "AAPL_POS", equity_aapl, 1.0, PositionUnit.UNITS),
    Position("POS_MSFT", "EQUITY_DESK", "MSFT_POS", equity_msft, 1.0, PositionUnit.UNITS),
    Position("POS_SWAP", "DERIVATIVES", "IRS_PAY_FIXED", swap_multi, 1.0, PositionUnit.UNITS),
]

diversified = (
    PortfolioBuilder("DIVERSIFIED_FUND")
    .name("Diversified Multi-Strategy Fund")
    .base_ccy(base_ccy)
    .as_of(as_of)
    .entity(entities)
    .position(positions_multi)
    .tag("fund_type", "hedge_fund")
    .build()
)

div_valuation = value_portfolio(diversified, market, FinstackConfig())

print(f"Total base value: {div_valuation.total_base_ccy.format()}")
print("Value by entity:")
for entity_id, value in div_valuation.by_entity.items():
    name = diversified.entities[entity_id].name or entity_id
    print(f"  {name}: {value.format()}")



Total base value: USD 6465095.39
Value by entity:
  Corporate Treasury: USD 4612385.57
  Equity Trading Desk: USD 1765000.00
  Derivatives Desk: USD 87709.82


## 4. Apply a rate shock scenario
If scenarios are enabled, apply a +50 bp parallel shift to the USD discount curve and revalue a simple bond position.



In [5]:
try:
    from finstack.portfolio import apply_and_revalue
    from finstack.scenarios import CurveKind, OperationSpec, ScenarioSpec
except ImportError:
    print("Scenarios feature not enabled - skipping scenario example.")
else:
    bond_shock = Bond.fixed_semiannual(
        "BOND_RATE_SENS",
        Money(10_000_000, "USD"),
        0.045,
        date(2024, 1, 15),
        date(2034, 1, 15),
        "USD-OIS",
    )

    treasury = Entity("TREASURY")
    shock_portfolio = (
        PortfolioBuilder("RATE_SENSITIVE")
        .base_ccy(base_ccy)
        .as_of(as_of)
        .entity(treasury)
        .position(Position("POS_BOND", treasury.id, "BOND_RATE_SENS", bond_shock, 1.0, PositionUnit.UNITS))
        .build()
    )

    base_val = value_portfolio(shock_portfolio, market)

    rate_shock = ScenarioSpec(
        "rate_shock_50bp",
        [OperationSpec.curve_parallel_bp(CurveKind.Discount, "USD-OIS", 50.0)],
        name="Rate Shock +50bp",
        description="Parallel shift of USD discount curve by 50 basis points",
    )

    shocked_val = apply_and_revalue(shock_portfolio, rate_shock, market)

    base_amount = base_val.total_base_ccy.amount
    shocked_amount = shocked_val.total_base_ccy.amount
    delta = shocked_amount - base_amount
    change_pct = (delta / base_amount) * 100 if base_amount != 0 else 0.0

    print(f"Base value:    {base_val.total_base_ccy.format()}")
    print(f"Shocked value: {shocked_val.total_base_ccy.format()}")
    print(f"Change:        {delta:,.2f} ({change_pct:+.2f}%)")



Base value:    USD 13269946.75
Shocked value: USD 12776319.90
Change:        -493,626.85 (-3.72%)


## 5. Position units and long/short handling
Positions can be expressed in face value, units, notionals with currency, or percentages. Negative quantities denote short exposure.



In [6]:
bond_face = Bond.fixed_semiannual(
    "BOND_FACE",
    Money(1_000_000, "USD"),
    0.050,
    date(2024, 1, 15),
    date(2029, 1, 15),
    "USD-OIS",
)

equity_units = Equity.create("EQUITY_UNITS", "AAPL", base_ccy, shares=100.0)

positions_units = [
    Position("POS_FACE", "PORTFOLIO_MGR", "BOND_FACE", bond_face, 10_000_000, PositionUnit.FACE_VALUE),
    Position("POS_UNITS", "PORTFOLIO_MGR", "EQUITY_UNITS", equity_units, 5_000.0, PositionUnit.UNITS),
    Position(
        "POS_NOTIONAL",
        "PORTFOLIO_MGR",
        "BOND_FACE",
        bond_face,
        5_000_000,
        PositionUnit.notional_with_ccy(base_ccy),
    ),
    Position("POS_PCT", "PORTFOLIO_MGR", "EQUITY_UNITS", equity_units, 0.001, PositionUnit.PERCENTAGE),
]

print("Position units demo:")
for pos in positions_units:
    direction = "LONG" if pos.is_long() else "SHORT"
    print(f"  {pos.position_id}: {direction} {pos.quantity} | unit={pos.unit}")

long_position = Position("POS_LONG", "HEDGE_FUND", "BOND_FACE", bond_face, 5.0, PositionUnit.UNITS)
short_position = Position("POS_SHORT", "HEDGE_FUND", "BOND_FACE", bond_face, -2.0, PositionUnit.UNITS)

print("\nLong/short demo:")
for pos in (long_position, short_position):
    direction = "LONG" if pos.is_long() else "SHORT"
    print(f"  {pos.position_id}: {direction} {abs(pos.quantity):.1f}x {pos.instrument_id}")



Position units demo:
  POS_FACE: LONG 10000000.0 | unit=face_value
  POS_UNITS: LONG 5000.0 | unit=units
  POS_NOTIONAL: LONG 5000000.0 | unit=notional(USD)
  POS_PCT: LONG 0.001 | unit=percentage

Long/short demo:
  POS_LONG: LONG 5.0x BOND_FACE
  POS_SHORT: SHORT 2.0x BOND_FACE


### Learning Objectives
- Set up reusable market data for portfolio valuation
- Build entities, instruments, and positions with the fluent `PortfolioBuilder`
- Value portfolios and aggregate metrics by position and entity
- Demonstrate multi-asset portfolios (rates + equities) and scenario shocks
- Inspect position units and long/short handling

